In [17]:
from __future__ import annotations
import math
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Tuple, Optional
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader
import zipfile
import requests
import io

## Rishi's part
data cleaning and preprocessing

In [18]:
def load_electricity_data():
  url = "https://archive.ics.uci.edu/static/public/321/electricityloaddiagrams20112014.zip"
  response = requests.get(url)

  z = zipfile.ZipFile(io.BytesIO(response.content))

  file_name = z.namelist()[0]

  df = pd.read_csv(
      z.open(file_name),
      sep=";",
      decimal=",",
      parse_dates=[0],
      quotechar='"'
  )

  df.rename(columns={df.columns[0]: "datetime"}, inplace=True)
  df.set_index("datetime", inplace=True)
  return df


In [19]:
def clean_data(df):
    df = df.copy()

    if isinstance(df.index, pd.DatetimeIndex):
        return df

    df.rename(columns={df.columns[0]: "datetime"}, inplace=True)
    df["datetime"] = pd.to_datetime(df["datetime"])
    df.set_index("datetime", inplace=True)

    return df

In [20]:
def aggregate_data(df):
  df["total_load"] = df.sum(axis=1)
  df_hourly = df["total_load"].resample("H").sum().to_frame()
  df_hourly.head()
  return df_hourly

In [21]:
def create_features(df_hourly):
  if isinstance(df_hourly, pd.Series):
      df_hourly = df_hourly.to_frame(name="total_load")

  df_hourly.index = pd.to_datetime(df_hourly.index)

  df_features = df_hourly.copy()

  df_features["hour_of_day"] = df_features.index.hour
  df_features["day_of_week"] = df_features.index.dayofweek
  df_features["is_weekend"] = (df_features["day_of_week"] >= 5).astype(int)
  df_features["month"] = df_features.index.month

  df_features["hour_sin"] = np.sin(2 * np.pi * df_features["hour_of_day"] / 24)
  df_features["hour_cos"] = np.cos(2 * np.pi * df_features["hour_of_day"] / 24)

  df_features["dayofweek_sin"] = np.sin(2 * np.pi * df_features["day_of_week"] / 7)
  df_features["dayofweek_cos"] = np.cos(2 * np.pi * df_features["day_of_week"] / 7)

  df_features["month_sin"] = np.sin(2 * np.pi * (df_features["month"] - 1) / 12)
  df_features["month_cos"] = np.cos(2 * np.pi * (df_features["month"] - 1) / 12)

  return df_features

In [22]:
df = load_electricity_data()
df = clean_data(df)
df_hourly = aggregate_data(df)
df= create_features(df_hourly)

/tmp/ipykernel_408/4251397852.py:3: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_hourly = df["total_load"].resample("H").sum().to_frame()


In [24]:
df.head()

,total_load,hour_of_day,day_of_week,is_weekend,month,hour_sin,hour_cos,dayofweek_sin,dayofweek_cos,month_sin,month_cos
datetime,,,,,,,,,,,
2011-01-01 00:00:00,207058.270272,0,5,1,1,0.000000,1.000000,-0.974928,-0.222521,0.0,1.0
2011-01-01 01:00:00,265378.510747,1,5,1,1,0.258819,0.965926,-0.974928,-0.222521,0.0,1.0
2011-01-01 02:00:00,263924.219533,2,5,1,1,0.500000,0.866025,-0.974928,-0.222521,0.0,1.0
2011-01-01 03:00:00,266306.134264,3,5,1,1,0.707107,0.707107,-0.974928,-0.222521,0.0,1.0
2011-01-01 04:00:00,259854.210701,4,5,1,1,0.866025,0.500000,-0.974928,-0.222521,0.0,1.0


## Baseline implementation

In [25]:
# ============================================================
# Config
# ============================================================

@dataclass
class Config:
    data_path: str = "hourly_aggregate_load.csv"
    timestamp_col: str = "timestamp"
    target_col: str = "total_load"

    input_window: int = 168       # past 168 hours = 7 days
    forecast_horizon: int = 24    # predict next 24 hours

    train_ratio: float = 0.70
    val_ratio: float = 0.15
    test_ratio: float = 0.15

    batch_size: int = 64
    hidden_size: int = 64
    num_layers: int = 2
    dropout: float = 0.1
    learning_rate: float = 1e-3
    num_epochs: int = 20
    patience: int = 5

    seasonal_lag_day: int = 24
    seasonal_lag_week: int = 24 * 7

    seed: int = 42
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


# Reproducibility

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)



In [26]:


# Metrics

def mae(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean(np.abs(y_true - y_pred)))


def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def mape(y_true: np.ndarray, y_pred: np.ndarray, eps: float = 1e-8) -> float:
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    return {
        "MAE": mae(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "MAPE": mape(y_true, y_pred),
    }



In [27]:

# Data loading and splitting


def load_hourly_data(path: str, timestamp_col: str, target_col: str) -> pd.DataFrame:
    df = pd.read_csv(path)

    if timestamp_col not in df.columns:
        raise ValueError(f"Missing timestamp column: {timestamp_col}")
    if target_col not in df.columns:
        raise ValueError(f"Missing target column: {target_col}")

    df[timestamp_col] = pd.to_datetime(df[timestamp_col])
    df = df.sort_values(timestamp_col).reset_index(drop=True)

    if df[target_col].isna().any():
        raise ValueError(f"Target column '{target_col}' contains missing values.")

    time_diffs = df[timestamp_col].diff().dropna()
    if not time_diffs.empty:
        most_common_diff = time_diffs.mode().iloc[0]
        if most_common_diff != pd.Timedelta(hours=1):
            print(
                f"[Warning] Most common time delta is {most_common_diff}, not 1 hour. "
                "Make sure Person 1 already resampled to hourly."
            )

    return df


def chronological_split(
    df: pd.DataFrame,
    train_ratio: float,
    val_ratio: float,
    test_ratio: float,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    total = len(df)
    if not math.isclose(train_ratio + val_ratio + test_ratio, 1.0, rel_tol=1e-6):
        raise ValueError("Train/val/test ratios must sum to 1.")

    train_end = int(total * train_ratio)
    val_end = train_end + int(total * val_ratio)

    train_df = df.iloc[:train_end].copy()
    val_df = df.iloc[train_end:val_end].copy()
    test_df = df.iloc[val_end:].copy()

    return train_df, val_df, test_df



In [28]:
# Baselines

def make_targets_for_eval(
    series: np.ndarray,
    input_window: int,
    forecast_horizon: int,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Creates aligned windows for evaluation.
    Returns:
        X_context: shape [num_samples, input_window]
        y_true:    shape [num_samples, forecast_horizon]
    """
    X_context, y_true = [], []
    max_start = len(series) - input_window - forecast_horizon + 1

    for start in range(max_start):
        in_end = start + input_window
        out_end = in_end + forecast_horizon

        X_context.append(series[start:in_end])
        y_true.append(series[in_end:out_end])

    return np.asarray(X_context), np.asarray(y_true)


def naive_last_value_forecast(
    X_context: np.ndarray,
    forecast_horizon: int,
) -> np.ndarray:
    """
    Repeat the last observed value across the forecast horizon.
    """
    last_vals = X_context[:, -1]
    return np.repeat(last_vals[:, None], forecast_horizon, axis=1)


def seasonal_naive_forecast(
    X_context: np.ndarray,
    forecast_horizon: int,
    seasonal_lag: int,
) -> np.ndarray:
    """
    Uses values from one seasonal cycle ago.
    Example:
      - seasonal_lag = 24   -> same hour previous day
      - seasonal_lag = 168  -> same hour previous week
    """
    if X_context.shape[1] < seasonal_lag:
        raise ValueError(
            f"Input window must be at least seasonal_lag={seasonal_lag}, "
            f"but got {X_context.shape[1]}"
        )

    preds = []
    for i in range(len(X_context)):
        context = X_context[i]
        pred = []
        for h in range(forecast_horizon):
            idx = -seasonal_lag + h
            if idx >= 0:
                idx = -1
            pred.append(context[idx])
        preds.append(pred)

    return np.asarray(preds)



In [29]:
# LSTM dataset

def build_supervised_sequences(
    series: np.ndarray,
    input_window: int,
    forecast_horizon: int,
) -> Tuple[np.ndarray, np.ndarray]:
    X, y = [], []
    max_start = len(series) - input_window - forecast_horizon + 1

    for start in range(max_start):
        in_end = start + input_window
        out_end = in_end + forecast_horizon

        X.append(series[start:in_end])
        y.append(series[in_end:out_end])

    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)

    return X, y


class SequenceDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray) -> None:
        self.X = torch.from_numpy(X).unsqueeze(-1)  # [N, seq_len, 1]
        self.y = torch.from_numpy(y)                # [N, horizon]

    def __len__(self) -> int:
        return len(self.X)

    def __getitem__(self, idx: int):
        return self.X[idx], self.y[idx]

# LSTM model

class LSTMForecaster(nn.Module):
    def __init__(
        self,
        input_size: int = 1,
        hidden_size: int = 64,
        num_layers: int = 2,
        dropout: float = 0.1,
        forecast_horizon: int = 24,
    ) -> None:
        super().__init__()

        lstm_dropout = dropout if num_layers > 1 else 0.0

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=lstm_dropout,
            batch_first=True,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, forecast_horizon)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [batch, seq_len, 1]
        out, _ = self.lstm(x)
        last_hidden = out[:, -1, :]       # [batch, hidden_size]
        last_hidden = self.dropout(last_hidden)
        preds = self.fc(last_hidden)      # [batch, horizon]
        return preds



In [30]:
# Train / eval loops

def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: str,
) -> float:
    model.train()
    total_loss = 0.0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        preds = model(X_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X_batch.size(0)

    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate_loss(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: str,
) -> float:
    model.eval()
    total_loss = 0.0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        preds = model(X_batch)
        loss = criterion(preds, y_batch)
        total_loss += loss.item() * X_batch.size(0)

    return total_loss / len(loader.dataset)


@torch.no_grad()
def predict_lstm(
    model: nn.Module,
    loader: DataLoader,
    device: str,
) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    preds_all, y_all = [], []

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        preds = model(X_batch).cpu().numpy()
        preds_all.append(preds)
        y_all.append(y_batch.numpy())

    return np.vstack(preds_all), np.vstack(y_all)


def inverse_transform_2d(
    scaler: StandardScaler,
    arr_2d: np.ndarray,
) -> np.ndarray:
    """
    Inverse transforms a [N, horizon] array using a scaler fit on a single feature.
    """
    flat = arr_2d.reshape(-1, 1)
    inv = scaler.inverse_transform(flat).reshape(arr_2d.shape)
    return inv



In [33]:
def main() -> None:
    cfg = Config()
    set_seed(cfg.seed)

    print(f"Using device: {cfg.device}")

    train_df, val_df, test_df = chronological_split(
        df,
        train_ratio=cfg.train_ratio,
        val_ratio=cfg.val_ratio,
        test_ratio=cfg.test_ratio,
    )

    print(f"Train rows: {len(train_df)}")
    print(f"Val rows:   {len(val_df)}")
    print(f"Test rows:  {len(test_df)}")

    train_series = train_df[cfg.target_col].to_numpy(dtype=np.float32)
    val_series = val_df[cfg.target_col].to_numpy(dtype=np.float32)
    test_series = test_df[cfg.target_col].to_numpy(dtype=np.float32)

    # -------------------------------
    # Baseline evaluation setup
    # For val/test baselines, prepend recent history from previous split
    # so each split has enough context for rolling forecasts.
    # -------------------------------
    val_context_series = np.concatenate([
        train_series[-cfg.input_window:],
        val_series
    ])
    test_context_series = np.concatenate([
        val_series[-cfg.input_window:],
        test_series
    ])

    X_val_ctx, y_val_true = make_targets_for_eval(
        val_context_series, cfg.input_window, cfg.forecast_horizon
    )
    X_test_ctx, y_test_true = make_targets_for_eval(
        test_context_series, cfg.input_window, cfg.forecast_horizon
    )

    # Remove any windows whose targets spill outside the actual split
    # Here, because we prepended only the context portion, these are already aligned.

    # -------------------------------
    # Naive baseline
    # -------------------------------
    val_pred_naive = naive_last_value_forecast(X_val_ctx, cfg.forecast_horizon)
    test_pred_naive = naive_last_value_forecast(X_test_ctx, cfg.forecast_horizon)

    # -------------------------------
    # Seasonal naive (previous day)
    # -------------------------------
    val_pred_seasonal_day = seasonal_naive_forecast(
        X_val_ctx, cfg.forecast_horizon, cfg.seasonal_lag_day
    )
    test_pred_seasonal_day = seasonal_naive_forecast(
        X_test_ctx, cfg.forecast_horizon, cfg.seasonal_lag_day
    )

    # -------------------------------
    # Seasonal naive (previous week)
    # -------------------------------
    val_pred_seasonal_week = seasonal_naive_forecast(
        X_val_ctx, cfg.forecast_horizon, cfg.seasonal_lag_week
    )
    test_pred_seasonal_week = seasonal_naive_forecast(
        X_test_ctx, cfg.forecast_horizon, cfg.seasonal_lag_week
    )

    # -------------------------------
    # Baseline metrics
    # Flatten across all windows and horizons
    # -------------------------------
    baseline_results = {
        "Naive_Last_Value_Val": compute_metrics(
            y_val_true.reshape(-1), val_pred_naive.reshape(-1)
        ),
        "Naive_Last_Value_Test": compute_metrics(
            y_test_true.reshape(-1), test_pred_naive.reshape(-1)
        ),
        "Seasonal_Naive_Day_Val": compute_metrics(
            y_val_true.reshape(-1), val_pred_seasonal_day.reshape(-1)
        ),
        "Seasonal_Naive_Day_Test": compute_metrics(
            y_test_true.reshape(-1), test_pred_seasonal_day.reshape(-1)
        ),
        "Seasonal_Naive_Week_Val": compute_metrics(
            y_val_true.reshape(-1), val_pred_seasonal_week.reshape(-1)
        ),
        "Seasonal_Naive_Week_Test": compute_metrics(
            y_test_true.reshape(-1), test_pred_seasonal_week.reshape(-1)
        ),
    }

    print("\n===== Baseline Results =====")
    for name, metrics in baseline_results.items():
        print(name, metrics)

    # =========================================================
    # LSTM preparation
    # =========================================================

    # Fit scaler ONLY on training data
    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_series.reshape(-1, 1)).reshape(-1)

    # For validation and test, include context from prior split
    val_scaled_full = scaler.transform(val_context_series.reshape(-1, 1)).reshape(-1)
    test_scaled_full = scaler.transform(test_context_series.reshape(-1, 1)).reshape(-1)

    # Build supervised sequences
    X_train, y_train = build_supervised_sequences(
        train_scaled, cfg.input_window, cfg.forecast_horizon
    )
    X_val, y_val = build_supervised_sequences(
        val_scaled_full, cfg.input_window, cfg.forecast_horizon
    )
    X_test, y_test = build_supervised_sequences(
        test_scaled_full, cfg.input_window, cfg.forecast_horizon
    )

    train_ds = SequenceDataset(X_train, y_train)
    val_ds = SequenceDataset(X_val, y_val)
    test_ds = SequenceDataset(X_test, y_test)

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=False)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False)

    # =========================================================
    # Train LSTM
    # =========================================================
    model = LSTMForecaster(
        input_size=1,
        hidden_size=cfg.hidden_size,
        num_layers=cfg.num_layers,
        dropout=cfg.dropout,
        forecast_horizon=cfg.forecast_horizon,
    ).to(cfg.device)

    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate)
    criterion = nn.MSELoss()

    best_val_loss = float("inf")
    best_state: Optional[dict] = None
    epochs_without_improvement = 0

    print("\n===== Training LSTM =====")
    for epoch in range(1, cfg.num_epochs + 1):
        train_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            device=cfg.device,
        )
        val_loss = evaluate_loss(
            model=model,
            loader=val_loader,
            criterion=criterion,
            device=cfg.device,
        )

        print(
            f"Epoch {epoch:02d} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= cfg.patience:
            print("Early stopping triggered.")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    # =========================================================
    # LSTM predictions
    # =========================================================
    val_pred_scaled, y_val_scaled = predict_lstm(model, val_loader, cfg.device)
    test_pred_scaled, y_test_scaled = predict_lstm(model, test_loader, cfg.device)

    val_pred_lstm = inverse_transform_2d(scaler, val_pred_scaled)
    y_val_lstm_true = inverse_transform_2d(scaler, y_val_scaled)

    test_pred_lstm = inverse_transform_2d(scaler, test_pred_scaled)
    y_test_lstm_true = inverse_transform_2d(scaler, y_test_scaled)

    lstm_results = {
        "LSTM_Val": compute_metrics(
            y_val_lstm_true.reshape(-1), val_pred_lstm.reshape(-1)
        ),
        "LSTM_Test": compute_metrics(
            y_test_lstm_true.reshape(-1), test_pred_lstm.reshape(-1)
        ),
    }

    print("\n===== LSTM Results =====")
    for name, metrics in lstm_results.items():
        print(name, metrics)

    # =========================================================
    # Save outputs for Person 4
    # =========================================================
    output_dir = Path("person2_outputs")
    output_dir.mkdir(parents=True, exist_ok=True)

    # Metrics summary
    metrics_rows = []
    for model_name, metrics in {**baseline_results, **lstm_results}.items():
        row = {"model": model_name}
        row.update(metrics)
        metrics_rows.append(row)

    metrics_df = pd.DataFrame(metrics_rows)
    metrics_df.to_csv(output_dir / "benchmark_metrics.csv", index=False)

    # Example saved predictions for downstream evaluation / plotting
    np.save(output_dir / "y_val_true_baselines.npy", y_val_true)
    np.save(output_dir / "y_test_true_baselines.npy", y_test_true)
    np.save(output_dir / "val_pred_naive.npy", val_pred_naive)
    np.save(output_dir / "test_pred_naive.npy", test_pred_naive)
    np.save(output_dir / "val_pred_seasonal_day.npy", val_pred_seasonal_day)
    np.save(output_dir / "test_pred_seasonal_day.npy", test_pred_seasonal_day)
    np.save(output_dir / "val_pred_seasonal_week.npy", val_pred_seasonal_week)
    np.save(output_dir / "test_pred_seasonal_week.npy", test_pred_seasonal_week)

    np.save(output_dir / "y_val_true_lstm.npy", y_val_lstm_true)
    np.save(output_dir / "y_test_true_lstm.npy", y_test_lstm_true)
    np.save(output_dir / "val_pred_lstm.npy", val_pred_lstm)
    np.save(output_dir / "test_pred_lstm.npy", test_pred_lstm)

    torch.save(model.state_dict(), output_dir / "lstm_model.pt")

    print(f"\nSaved outputs to: {output_dir.resolve()}")

In [34]:
main()

Using device: cuda
Train rows: 24545
Val rows:   5259
Test rows:  5261

===== Baseline Results =====
Naive_Last_Value_Val {'MAE': 255663.390625, 'RMSE': 327192.375, 'MAPE': 44.57447052001953}
Naive_Last_Value_Test {'MAE': 350945.1875, 'RMSE': 461291.125, 'MAPE': 42.96784210205078}
Seasonal_Naive_Day_Val {'MAE': 28049.154296875, 'RMSE': 57315.078125, 'MAPE': 9.243870735168457}
Seasonal_Naive_Day_Test {'MAE': 35075.90234375, 'RMSE': 57475.1953125, 'MAPE': 3.774121046066284}
Seasonal_Naive_Week_Val {'MAE': 35390.171875, 'RMSE': 63164.265625, 'MAPE': 9.629261016845703}
Seasonal_Naive_Week_Test {'MAE': 60001.73046875, 'RMSE': 95530.203125, 'MAPE': 5.80952787399292}

===== Training LSTM =====
Epoch 01 | Train Loss: 0.225172 | Val Loss: 0.720665
Epoch 02 | Train Loss: 0.300074 | Val Loss: 0.817715
Epoch 03 | Train Loss: 0.309917 | Val Loss: 0.607631
Epoch 04 | Train Loss: 0.212193 | Val Loss: 0.432335
Epoch 05 | Train Loss: 0.110611 | Val Loss: 0.343699
Epoch 06 | Train Loss: 0.073888 | Val L